# Version 22 : Triple Ensemble Optimisé (XGBoost 6.1 + CoxNet 21.1 + DeepSurv 20.1)

**Ensemble Final** : 3 modèles state-of-the-art avec features enrichies
- 🚀 **XGBoost 6.1**: C-index 0.7413, features auto-sélectionnées
- 📊 **CoxNet 21.1 Enriched**: C-index 0.7493, 95/132 features, L1_ratio=0.3, alpha=0.01
- 🧠 **DeepSurv 20.1 Improved**: C-index 0.7466, [256,128,64,32] + residual connections

**Optimisation** :
- Poids optimisés via Optuna
- Features enrichies (transformations, ratios, interactions)
- Maximum de diversité : Linear (CoxNet) + Tree (XGBoost) + NN (DeepSurv)

**Objectif** : C-index > 0.75

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import os

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Survival
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Seeds
np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported")
print(f"PyTorch version: {torch.__version__}")
print(f"XGBoost version: {xgb.__version__}")

✓ Libraries imported
PyTorch version: 2.9.1+cpu
XGBoost version: 2.1.4


## 2. Data Loading

In [2]:
DATA_PATH = "C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(os.path.join(DATA_PATH, "X_train", "clinical_train.csv"))
target_train = pd.read_csv(os.path.join(DATA_PATH, "target_train.csv"))
clinical_test = pd.read_csv(os.path.join(DATA_PATH, "X_test", "clinical_test.csv"))
molecular_train = pd.read_csv(os.path.join(DATA_PATH, "X_train", "molecular_train.csv"))
molecular_test = pd.read_csv(os.path.join(DATA_PATH, "X_test", "molecular_test.csv"))

print(f"✓ Data loaded: {clinical_train.shape[0]} train patients")

✓ Data loaded: 3323 train patients


## 3. Feature Engineering (Unified for all models)

In [3]:
# Reuse feature engineering from V21.1 (enriched features)
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    
    # Chromosomes
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

def add_enriched_features(X_clinical, X_molecular):
    """Enriched features from V21.1"""
    X_enriched = pd.DataFrame(index=X_clinical.index)
    
    # Non-linear transformations
    for col in ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES']:
        if col in X_clinical.columns:
            X_enriched[f'{col}_log'] = np.log1p(X_clinical[col])
            X_enriched[f'{col}_sqrt'] = np.sqrt(X_clinical[col].clip(lower=0))
            X_enriched[f'{col}_squared'] = X_clinical[col] ** 2
    
    # Ratios
    if 'WBC' in X_clinical.columns and 'ANC' in X_clinical.columns:
        X_enriched['WBC_ANC_ratio'] = X_clinical['WBC'] / (X_clinical['ANC'] + 1e-5)
    if 'HB' in X_clinical.columns and 'PLT' in X_clinical.columns:
        X_enriched['HB_PLT_ratio'] = X_clinical['HB'] / (X_clinical['PLT'] + 1e-5)
    if 'WBC' in X_clinical.columns and 'MONOCYTES' in X_clinical.columns:
        X_enriched['WBC_MONOCYTES_ratio'] = X_clinical['WBC'] / (X_clinical['MONOCYTES'] + 1e-5)
    if 'BM_BLAST' in X_clinical.columns and 'WBC' in X_clinical.columns:
        X_enriched['BLAST_WBC_ratio'] = X_clinical['BM_BLAST'] / (X_clinical['WBC'] + 1e-5)
    
    # Interactions
    important_pairs = [
        ('BM_BLAST', 'WBC'), ('BM_BLAST', 'HB'), ('BM_BLAST', 'PLT'),
        ('WBC', 'HB'), ('HB', 'PLT'), ('ANC', 'MONOCYTES')
    ]
    for col1, col2 in important_pairs:
        if col1 in X_clinical.columns and col2 in X_clinical.columns:
            X_enriched[f'{col1}_x_{col2}'] = X_clinical[col1] * X_clinical[col2]
    
    # Molecular × Clinical
    if 'mutation_count_total' in X_molecular.columns:
        if 'BM_BLAST' in X_clinical.columns:
            X_enriched['mutations_x_BLAST'] = X_molecular['mutation_count_total'] * X_clinical['BM_BLAST']
        if 'WBC' in X_clinical.columns:
            X_enriched['mutations_x_WBC'] = X_molecular['mutation_count_total'] * X_clinical['WBC']
    if 'vaf_sum' in X_molecular.columns and 'BM_BLAST' in X_clinical.columns:
        X_enriched['vaf_sum_x_BLAST'] = X_molecular['vaf_sum'] * X_clinical['BM_BLAST']
    
    # Binning
    if 'BM_BLAST' in X_clinical.columns:
        X_enriched['BLAST_low'] = (X_clinical['BM_BLAST'] < 20).astype(int)
        X_enriched['BLAST_high'] = (X_clinical['BM_BLAST'] >= 50).astype(int)
    
    return X_enriched

print("✓ Feature engineering functions defined")

✓ Feature engineering functions defined


## 4. Build Features

In [4]:
# Build unified feature set
target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_clinical_train = clinical_train_clean[numeric_features].copy()
center_encoded_train = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical_train = pd.concat([X_clinical_train, center_encoded_train], axis=1)

cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train_clean.index.unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids, top_n_genes=20)
mol_features_train_aligned = mol_features_train.reindex(X_clinical_train.index, fill_value=0)

enriched_features_train = add_enriched_features(X_clinical_train[numeric_features], mol_features_train_aligned)

cyto_features_train_aligned = cyto_features_train.reindex(X_clinical_train.index, fill_value=0)
X_combined_train = pd.concat([
    X_clinical_train, mol_features_train_aligned, cyto_features_train_aligned, enriched_features_train
], axis=1)

# Test set
X_clinical_test = clinical_test.set_index('ID')[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test.set_index('ID')['CENTER'], prefix='CENTER', drop_first=True)
X_clinical_test = pd.concat([X_clinical_test, center_encoded_test], axis=1)

for col in X_clinical_train.columns:
    if col not in X_clinical_test.columns:
        X_clinical_test[col] = 0
X_clinical_test = X_clinical_test[X_clinical_train.columns]

cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids, top_n_genes=20)

for col in mol_features_train.columns:
    if col not in mol_features_test.columns:
        mol_features_test[col] = 0
mol_features_test = mol_features_test[mol_features_train.columns]

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)
enriched_features_test = add_enriched_features(X_clinical_test[numeric_features], mol_features_test_aligned)
cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)

X_combined_test = pd.concat([
    X_clinical_test, mol_features_test_aligned, cyto_features_test_aligned, enriched_features_test
], axis=1)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

print(f"✓ Features created: {X_combined_train.shape}")
print(f"  Enriched features: ~{enriched_features_train.shape[1]}")

✓ Features created: (3173, 159)
  Enriched features: ~27


## 5. Train/Val Split

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X_combined_train, y_surv, test_size=0.2, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print(f"✓ Split: {X_train.shape[0]} train, {X_val.shape[0]} val")

✓ Split: 2538 train, 635 val


## 6. MODEL 1: XGBoost 6.1

In [6]:
print("="*60)
print("MODEL 1: XGBOOST 6.1 (from V10)")
print("="*60)

# XGBoost params from V6.1
XGB_PARAMS = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'learning_rate': 0.0744,
    'max_depth': 3,
    'min_child_weight': 9,
    'subsample': 0.6194,
    'colsample_bytree': 0.8872,
    'reg_alpha': 0.0124,
    'reg_lambda': 0.0043,
    'gamma': 0.0600,
}

# Impute for XGBoost
imputer_xgb = SimpleImputer(strategy='median')
X_train_xgb = pd.DataFrame(
    imputer_xgb.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)
X_val_xgb = pd.DataFrame(
    imputer_xgb.transform(X_val),
    index=X_val.index,
    columns=X_val.columns
)

# Prepare labels
y_train_xgb = y_train['OS_YEARS'].copy()
y_train_xgb[~y_train['OS_STATUS']] = -y_train_xgb[~y_train['OS_STATUS']]

y_val_xgb = y_val['OS_YEARS'].copy()
y_val_xgb[~y_val['OS_STATUS']] = -y_val_xgb[~y_val['OS_STATUS']]

dtrain = xgb.DMatrix(X_train_xgb, label=y_train_xgb)
dval = xgb.DMatrix(X_val_xgb, label=y_val_xgb)

# Train
xgb_model = xgb.train(
    XGB_PARAMS,
    dtrain,
    num_boost_round=500,
    evals=[(dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=False
)

y_pred_xgb_val = xgb_model.predict(dval)
c_index_xgb = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_xgb_val
)[0]

print(f"✓ XGBoost trained")
print(f"  C-index: {c_index_xgb:.4f}")
print(f"  Best iteration: {xgb_model.best_iteration}")

MODEL 1: XGBOOST 6.1 (from V10)
✓ XGBoost trained
  C-index: 0.7508
  Best iteration: 72


## 7. MODEL 2: CoxNet 21.1 Enriched

In [7]:
print("\n" + "="*60)
print("MODEL 2: COXNET 21.1 ENRICHED")
print("="*60)

# Same preprocessing as V21.1
imputer_cox = SimpleImputer(strategy='median')
X_train_cox = pd.DataFrame(
    imputer_cox.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)
X_val_cox = pd.DataFrame(
    imputer_cox.transform(X_val),
    index=X_val.index,
    columns=X_val.columns
)

# Scale (CRITICAL for CoxNet)
scaler_cox = StandardScaler()
X_train_cox_scaled = pd.DataFrame(
    scaler_cox.fit_transform(X_train_cox),
    index=X_train_cox.index,
    columns=X_train_cox.columns
)
X_val_cox_scaled = pd.DataFrame(
    scaler_cox.transform(X_val_cox),
    index=X_val_cox.index,
    columns=X_val_cox.columns
)

# Train CoxNet with best params from V21.1
cox_model = CoxnetSurvivalAnalysis(
    l1_ratio=0.3,  # Best from V21.1
    alphas=[0.01],  # Best from V21.1
    max_iter=100000,
    fit_baseline_model=True
)

cox_model.fit(X_train_cox_scaled.values, y_train)

y_pred_cox_val = cox_model.predict(X_val_cox_scaled.values)
c_index_cox = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_cox_val
)[0]

n_selected_cox = np.sum(cox_model.coef_.flatten() != 0)
print(f"✓ CoxNet trained")
print(f"  C-index: {c_index_cox:.4f}")
print(f"  Features selected: {n_selected_cox}/{X_train.shape[1]}")


MODEL 2: COXNET 21.1 ENRICHED
✓ CoxNet trained
  C-index: 0.7399
  Features selected: 121/159


## 8. MODEL 3: DeepSurv 20.1 Improved

In [8]:
print("\n" + "="*60)
print("MODEL 3: DEEPSURV 20.1 IMPROVED")
print("="*60)

# Same preprocessing as DeepSurv
X_train_nn_scaled = X_train_cox_scaled.copy()  # Reuse scaling from CoxNet
X_val_nn_scaled = X_val_cox_scaled.copy()

# PyTorch Dataset
class SurvivalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.event = torch.FloatTensor(y['OS_STATUS'].astype(float))
        self.time = torch.FloatTensor(np.array(y['OS_YEARS'], dtype=np.float32))
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.event[idx], self.time[idx]

# Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.4):
        super(ResidualBlock, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.bn = nn.BatchNorm1d(output_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()
    
    def forward(self, x):
        identity = self.skip(x)
        out = self.linear(x)
        out = self.bn(out)
        out = self.activation(out)
        out = self.dropout(out)
        out = out + identity
        return out

# Improved DeepSurv
class ImprovedDeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_layers=[256, 128, 64, 32], dropout=0.4):
        super(ImprovedDeepSurv, self).__init__()
        self.input_layer = ResidualBlock(input_dim, hidden_layers[0], dropout)
        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_layers)-1):
            self.hidden_layers.append(
                ResidualBlock(hidden_layers[i], hidden_layers[i+1], dropout)
            )
        self.output = nn.Linear(hidden_layers[-1], 1)
    
    def forward(self, x):
        x = self.input_layer(x)
        for layer in self.hidden_layers:
            x = layer(x)
        return self.output(x)

# Training functions
def cox_ph_loss(risk_scores, events, times):
    sorted_indices = torch.argsort(times, descending=True)
    risk_scores = risk_scores[sorted_indices].squeeze()
    events = events[sorted_indices]
    hazard_ratio = torch.exp(risk_scores)
    log_risk = torch.log(torch.cumsum(hazard_ratio, dim=0) + 1e-7)
    uncensored_likelihood = risk_scores - log_risk
    loss = -torch.sum(uncensored_likelihood * events) / (torch.sum(events) + 1e-7)
    return loss

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, event_batch, time_batch in dataloader:
        X_batch = X_batch.to(device)
        event_batch = event_batch.to(device)
        time_batch = time_batch.to(device)
        optimizer.zero_grad()
        risk_scores = model(X_batch)
        loss = cox_ph_loss(risk_scores, event_batch, time_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_risk_scores = []
    all_events = []
    all_times = []
    with torch.no_grad():
        for X_batch, event_batch, time_batch in dataloader:
            X_batch = X_batch.to(device)
            risk_scores = model(X_batch)
            all_risk_scores.extend(risk_scores.cpu().numpy())
            all_events.extend(event_batch.cpu().numpy())
            all_times.extend(time_batch.cpu().numpy())
    
    all_risk_scores = np.array(all_risk_scores).flatten()
    all_events = np.array(all_events).astype(bool)
    all_times = np.array(all_times)
    c_index = concordance_index_censored(all_events, all_times, all_risk_scores)[0]
    return c_index

# Create datasets
train_dataset = SurvivalDataset(X_train_nn_scaled, y_train)
val_dataset = SurvivalDataset(X_val_nn_scaled, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_nn = ImprovedDeepSurv(input_dim=X_train.shape[1], hidden_layers=[256, 128, 64, 32], dropout=0.4)
model_nn = model_nn.to(device)

# Train
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.0005, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

best_val_c_index = 0
patience_counter = 0
patience = 20

print(f"  Training on {device}...")
for epoch in range(150):
    train_loss = train_epoch(model_nn, train_loader, optimizer, device)
    val_c_index = evaluate(model_nn, val_loader, device)
    
    scheduler.step(train_loss)
    
    if val_c_index > best_val_c_index:
        best_val_c_index = val_c_index
        best_model_state = model_nn.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"  Early stopping at epoch {epoch+1}")
        break

model_nn.load_state_dict(best_model_state)

# Get predictions
model_nn.eval()
with torch.no_grad():
    X_val_tensor = torch.FloatTensor(X_val_nn_scaled.values).to(device)
    y_pred_nn_val = model_nn(X_val_tensor).cpu().numpy().flatten()

c_index_nn = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_nn_val
)[0]

print(f"\n✓ DeepSurv trained")
print(f"  C-index: {c_index_nn:.4f}")


MODEL 3: DEEPSURV 20.1 IMPROVED
  Training on cpu...
  Early stopping at epoch 51

✓ DeepSurv trained
  C-index: 0.7393


## 9. Optimize Ensemble Weights

In [9]:
print("\n" + "="*60)
print("OPTIMIZING ENSEMBLE WEIGHTS (3 MODELS)")
print("="*60)

def objective(trial):
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_cox = trial.suggest_float('w_cox', 0.0, 1.0 - w_xgb)
    w_nn = 1 - w_xgb - w_cox
    
    y_pred_ensemble = (
        w_xgb * y_pred_xgb_val + 
        w_cox * y_pred_cox_val + 
        w_nn * y_pred_nn_val
    )
    
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_ensemble
    )[0]
    
    return c_index

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200, show_progress_bar=True)

best_w_xgb = study.best_params['w_xgb']
best_w_cox = study.best_params['w_cox']
best_w_nn = 1 - best_w_xgb - best_w_cox
best_c_index = study.best_value

print(f"\n✓ Optimization complete")
print(f"\nBest weights:")
print(f"  XGBoost:  {best_w_xgb:.3f}")
print(f"  CoxNet:   {best_w_cox:.3f}")
print(f"  DeepSurv: {best_w_nn:.3f}")
print(f"\nBest ensemble C-index: {best_c_index:.4f}")


OPTIMIZING ENSEMBLE WEIGHTS (3 MODELS)


  0%|          | 0/200 [00:00<?, ?it/s]


✓ Optimization complete

Best weights:
  XGBoost:  0.537
  CoxNet:   0.097
  DeepSurv: 0.366

Best ensemble C-index: 0.7570


## 10. Performance Comparison

In [10]:
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

comparison = pd.DataFrame([
    {'Model': 'XGBoost 6.1', 'Type': 'Tree', 'C-index': c_index_xgb},
    {'Model': 'CoxNet 21.1', 'Type': 'Linear', 'C-index': c_index_cox},
    {'Model': 'DeepSurv 20.1', 'Type': 'Neural Net', 'C-index': c_index_nn},
    {'Model': 'V22 Ensemble', 'Type': 'Ensemble', 'C-index': best_c_index}
])

print("\n" + comparison.to_string(index=False))

improvement = best_c_index - max(c_index_xgb, c_index_cox, c_index_nn)
print(f"\n📈 Ensemble improvement: +{improvement:.4f}")
print(f"\n🎯 Target: 0.75+")
print(f"   Achieved: {best_c_index:.4f}")


PERFORMANCE COMPARISON

        Model       Type  C-index
  XGBoost 6.1       Tree 0.750791
  CoxNet 21.1     Linear 0.739937
DeepSurv 20.1 Neural Net 0.739257
 V22 Ensemble   Ensemble 0.756963

📈 Ensemble improvement: +0.0062

🎯 Target: 0.75+
   Achieved: 0.7570


## 11. Generate Final Predictions

In [11]:
print("\n" + "="*60)
print("GENERATING FINAL PREDICTIONS")
print("="*60)

# Preprocess test set for each model
# XGBoost
X_test_xgb = pd.DataFrame(
    imputer_xgb.transform(X_combined_test),
    index=X_combined_test.index,
    columns=X_combined_test.columns
)
dtest = xgb.DMatrix(X_test_xgb)
y_pred_xgb_test = xgb_model.predict(dtest)

# CoxNet
X_test_cox = pd.DataFrame(
    imputer_cox.transform(X_combined_test),
    index=X_combined_test.index,
    columns=X_combined_test.columns
)
X_test_cox_scaled = pd.DataFrame(
    scaler_cox.transform(X_test_cox),
    index=X_test_cox.index,
    columns=X_test_cox.columns
)
y_pred_cox_test = cox_model.predict(X_test_cox_scaled.values)

# DeepSurv
model_nn.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test_cox_scaled.values).to(device)
    y_pred_nn_test = model_nn(X_test_tensor).cpu().numpy().flatten()

# Ensemble
y_pred_ensemble_test = (
    best_w_xgb * y_pred_xgb_test + 
    best_w_cox * y_pred_cox_test + 
    best_w_nn * y_pred_nn_test
)

# Create submission
submission = pd.DataFrame({
    'ID': X_combined_test.index,
    'OS_YEARS': y_pred_ensemble_test
})

submission_path = os.path.join(DATA_PATH, "submission_v22_triple_ensemble.csv")
submission.to_csv(submission_path, index=False)

print(f"\n✓ Submission created: {submission_path}")
print(f"\nPreview:")
print(submission.head(10))
print(f"\nRisk score statistics:")
print(submission['OS_YEARS'].describe())


GENERATING FINAL PREDICTIONS

✓ Submission created: C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie\submission_v22_triple_ensemble.csv

Preview:
      ID  OS_YEARS
0   KYW1  1.403383
1   KYW2  5.801590
2   KYW3  0.594443
3   KYW4  0.832216
4   KYW5  3.765941
5   KYW6  4.044163
6   KYW7  2.260749
7   KYW8  1.598167
8   KYW9 -0.315833
9  KYW10 -0.456741

Risk score statistics:
count     1193.000000
mean        78.692234
std       2637.102372
min         -3.440769
25%          0.028891
50%          1.161549
75%          2.808892
max      91087.307614
Name: OS_YEARS, dtype: float64


## 12. Summary

In [12]:
print("\n" + "="*60)
print("SUMMARY - V22 TRIPLE ENSEMBLE")
print("="*60)

print(f"\n🎯 Ensemble Configuration:")
print(f"  XGBoost 6.1:     {best_w_xgb:.1%} (C-index: {c_index_xgb:.4f})")
print(f"  CoxNet 21.1:     {best_w_cox:.1%} (C-index: {c_index_cox:.4f})")
print(f"  DeepSurv 20.1:   {best_w_nn:.1%} (C-index: {c_index_nn:.4f})")

print(f"\n📊 Performance:")
print(f"  Best single model: {max(c_index_xgb, c_index_cox, c_index_nn):.4f}")
print(f"  Ensemble:          {best_c_index:.4f}")
print(f"  Improvement:       +{improvement:.4f}")

print(f"\n✨ Key Features:")
print(f"  - Enriched features: {enriched_features_train.shape[1]}")
print(f"  - Total features: {X_combined_train.shape[1]}")
print(f"  - Model diversity: Linear + Tree + Neural")

print(f"\n📁 Output:")
print(f"  {submission_path}")
print("="*60)


SUMMARY - V22 TRIPLE ENSEMBLE

🎯 Ensemble Configuration:
  XGBoost 6.1:     53.7% (C-index: 0.7508)
  CoxNet 21.1:     9.7% (C-index: 0.7399)
  DeepSurv 20.1:   36.6% (C-index: 0.7393)

📊 Performance:
  Best single model: 0.7508
  Ensemble:          0.7570
  Improvement:       +0.0062

✨ Key Features:
  - Enriched features: 27
  - Total features: 159
  - Model diversity: Linear + Tree + Neural

📁 Output:
  C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie\submission_v22_triple_ensemble.csv
